# MobileADAS3D-S1-V2 continuous-yaw gate

Fresh GT-only controlled experiment. The only modeling change from rejected S1-V1 is replacing axis plus hard direction with normalized `[sin(yaw), cos(yaw)]`. Select a GPU runtime. Do not resume an S1-V1 checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime
from collections import deque
import json, os, shlex, subprocess, sys
REPO_URL='https://github.com/Ali-RT/mobile_adas3d.git'
BRANCH='main'
PROJECT_DIR=Path('/content/mobile_adas3d')
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti')
DATASET_VIEW=Path('/content/kitti_s1_v2')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
TAXONOMY_MANIFEST=Path('/content/drive/MyDrive/mobile_adas3d_manifests/kitti_s1_taxonomy_manifest.json')
OUTPUT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_outputs/mobileadas3d_s1_v2_continuous_yaw')
R0_SELECTION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
BASE_CONFIG=PROJECT_DIR/'configs/kitti_mobileadas3d_s1_v2_continuous_yaw.yaml'
RUNTIME_CONFIG_DIR=PROJECT_DIR/'configs/runtime_s1_v2'
RUN_NAME='mobileadas3d_s1_v2_continuous_yaw'
GATE_EPOCHS=20
FULL_EPOCHS=100
def run(command,cwd=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    result=subprocess.run(command,cwd=cwd)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_streamed(command,cwd,log_path):
    command=[str(x) for x in command]; log_path.parent.mkdir(parents=True,exist_ok=True)
    print('+',shlex.join(command)); print('Durable log:',log_path)
    tail=deque(maxlen=120); env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        p=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=p.wait()
    if code: raise RuntimeError(f'Exit {code}; log={log_path}\n'+'\n'.join(tail))

In [ ]:
# Repository, dependencies, and GPU. Push the required local commits before running.
if not (PROJECT_DIR/'.git').exists(): run(['git','clone','--branch',BRANCH,REPO_URL,PROJECT_DIR])
else:
    run(['git','fetch','origin'],cwd=PROJECT_DIR)
    run(['git','checkout',BRANCH],cwd=PROJECT_DIR)
    run(['git','pull','--ff-only','origin',BRANCH],cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],cwd=PROJECT_DIR)
import torch, timm
if not torch.cuda.is_available(): raise RuntimeError('Choose Runtime > Change runtime type > GPU')
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_DIR,text=True).strip())
print('torch/timm:',torch.__version__,timm.__version__)
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
# Prefer complete local KITTI; otherwise create a zero-copy canonical Drive view.
def count_files(path,suffix): return sum(1 for p in path.iterdir() if p.is_file() and p.suffix==suffix) if path.is_dir() else 0
def complete(root): return count_files(root/'training/image_2','.png')==7481 and count_files(root/'training/label_2','.txt')==7481 and count_files(root/'training/calib','.txt')==7481
if complete(LOCAL_DATASET_ROOT): DATASET_ROOT=LOCAL_DATASET_ROOT
else:
    aliases={'image_2':['image_2','image_02'],'label_2':['label_2','label_02'],'calib':['calib']}
    (DATASET_VIEW/'training').mkdir(parents=True,exist_ok=True)
    for canonical,candidates in aliases.items():
        source=next((DRIVE_DATASET_ROOT/'training'/name for name in candidates if (DRIVE_DATASET_ROOT/'training'/name).is_dir()),None)
        if source is None: raise FileNotFoundError(f'Missing source for {canonical}')
        link=DATASET_VIEW/'training'/canonical
        if link.is_symlink() and link.resolve()==source.resolve(): continue
        if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace unexpected {link}')
        link.symlink_to(source,target_is_directory=True)
    DATASET_ROOT=DATASET_VIEW
if not complete(DATASET_ROOT): raise RuntimeError(f'Incomplete KITTI root: {DATASET_ROOT}')
print('Dataset root:',DATASET_ROOT)

In [ ]:
# Freeze R0 provenance, generate V2 runtime configs, refresh taxonomy, and run a real CUDA loss.
if not R0_SELECTION.is_file(): raise FileNotFoundError(R0_SELECTION)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
run([sys.executable,'scripts/prepare_s1_gt_baseline.py','--base-config',BASE_CONFIG,'--r0-selection',R0_SELECTION,'--output-dir',OUTPUT_DIR,'--config-dir',RUNTIME_CONFIG_DIR,'--run-name',RUN_NAME,'--gate-epochs',GATE_EPOCHS,'--full-epochs',FULL_EPOCHS],cwd=PROJECT_DIR)
GATE_CONFIG=RUNTIME_CONFIG_DIR/'mobileadas3d_s1_gt_gate20.yaml'
FULL_CONFIG=RUNTIME_CONFIG_DIR/'mobileadas3d_s1_gt_full100.yaml'
COMMON=['--profile','colab_drive','--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR,'--output-dir',OUTPUT_DIR]
run_streamed([sys.executable,'-u','scripts/create_kitti_taxonomy_manifest.py','--config',GATE_CONFIG,'--profile','colab_drive','--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR,'--output',TAXONOMY_MANIFEST],PROJECT_DIR,OUTPUT_DIR/'taxonomy_audit.log')
run_streamed([sys.executable,'-u','scripts/check_training_ready.py','--config',GATE_CONFIG,*COMMON,'--require-cuda','--report',OUTPUT_DIR/'training_preflight.json'],PROJECT_DIR,OUTPUT_DIR/'training_preflight.log')
manifest=json.loads((OUTPUT_DIR/'s1_gt_baseline_manifest.json').read_text())
assert manifest['distillation_enabled'] is False
assert manifest['yaw_encoding']=='continuous_sincos'
print(json.dumps(manifest,indent=2))

## Fresh 20-epoch S1-V2 health gate

This is real training. Re-running resumes only the newest matching V2 run. The dedicated output directory prevents accidental S1-V1 resume.

In [ ]:
candidates=sorted((OUTPUT_DIR/'runs').glob(f'*{RUN_NAME}*/checkpoints/latest.pt'),key=lambda p:p.stat().st_mtime)
RESUME=candidates[-1] if candidates else None
if RESUME:
    payload=torch.load(RESUME,map_location='cpu',weights_only=False)
    saved_yaw=payload.get('config',{}).get('model',{}).get('yaw_encoding')
    if saved_yaw!='continuous_sincos': raise RuntimeError(f'Refusing incompatible resume yaw={saved_yaw!r}: {RESUME}')
    if payload.get('epoch',0)>=GATE_EPOCHS: print(f'Gate already complete at epoch {payload["epoch"]}: {RESUME}')
    else: run_streamed([sys.executable,'-u','scripts/train_mobile_adas3d.py','--config',GATE_CONFIG,*COMMON,'--run-name',RUN_NAME,'--resume',RESUME],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/f'gate_resume_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
else:
    run_streamed([sys.executable,'-u','scripts/train_mobile_adas3d.py','--config',GATE_CONFIG,*COMMON,'--run-name',RUN_NAME],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/f'gate_fresh_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
candidates=sorted((OUTPUT_DIR/'runs').glob(f'*{RUN_NAME}*/checkpoints/latest.pt'),key=lambda p:p.stat().st_mtime)
LATEST_CHECKPOINT=candidates[-1]; TRAIN_RUN_DIR=LATEST_CHECKPOINT.parent.parent
print('Gate checkpoint:',LATEST_CHECKPOINT); print('Run directory:',TRAIN_RUN_DIR)

In [ ]:
# Full product AP and geometry/yaw evaluation of epoch 20.
GATE_EVAL_DIR=TRAIN_RUN_DIR/'kitti_r40_gate20'
run_streamed([sys.executable,'-u','scripts/evaluate_kitti_r40.py','--config',GATE_CONFIG,*COMMON,'--checkpoint',LATEST_CHECKPOINT,'--split','val','--score-threshold','0.001','--topk','300','--nms-iou-threshold','0.5','--output-dir',GATE_EVAL_DIR],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/'gate20_product_eval.log')
import pandas as pd
gate_summary=json.loads((GATE_EVAL_DIR/'kitti_r40_summary.json').read_text())
assert gate_summary['complete_split'] and gate_summary['evaluated_images']==3769
display(pd.DataFrame(gate_summary['metrics']).pivot_table(index=['metric','class_name'],columns='difficulty',values='ap_r40').round(3))
GEOMETRY_DIR=TRAIN_RUN_DIR/'geometry_diagnostic_epoch20'
run_streamed([sys.executable,'-u','scripts/evaluate_3d_metrics.py','--config',GATE_CONFIG,'--profile','colab_drive','--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR,'--checkpoint',LATEST_CHECKPOINT,'--split','val','--score-threshold','0.001','--match-iou-threshold','0.5','--topk','300','--nms-iou-threshold','0.5','--output-dir',GEOMETRY_DIR],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/'gate20_geometry.log')
summary3d=pd.read_csv(GEOMETRY_DIR/'summary_3d_metrics_val.csv')
display(summary3d[summary3d.group_name.eq('class_name')][['group_value','count','iou_2d_mean','depth_rel_error_mean','yaw_abs_error_mean_deg','dim_mae_m','center3d_mae_m','corner3d_mae_m']].round(3))
print('Send the AP table, final training summary, and per-class geometry table before continuation.')

## Locked continuation

Do not continue until the complete epoch-20 comparison against S1-V1 is reviewed.

In [ ]:
AUTHORIZE_CONTINUATION=False
if not AUTHORIZE_CONTINUATION: raise RuntimeError('Review S1-V2 epoch-20 gate before continuation')
run_streamed([sys.executable,'-u','scripts/train_mobile_adas3d.py','--config',FULL_CONFIG,*COMMON,'--run-name',RUN_NAME,'--resume',LATEST_CHECKPOINT],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/f'full_resume_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')